In [ ]:
import sys
import os

from opt_einsum import contract
sys.path.append(os.path.abspath('..'))  # Add parent directory to path

import numpy as np
from diffusion_geometry.visualisation import *
from plotly.subplots import make_subplots
from diffusion_geometry import Function, VectorField, Form
from generate_data import gen_2d_data
from diffusion_geometry import DiffusionGeometry
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.textpath import TextPath
from matplotlib.font_manager import FontProperties
from matplotlib.path import Path


In [ ]:



def cov_der(dg, X, Y):
    return dg.levi_civita(Y)(X)


def sample_perturbed_grid(range_x, range_y, num_x, num_y, noise_strength=0.3):
    x_lin = np.linspace(range_x[0], range_x[1], num_x)
    y_lin = np.linspace(range_y[0], range_y[1], num_y)
    x_grid, y_grid = np.meshgrid(x_lin, y_lin)
    data = np.column_stack([x_grid.ravel(), y_grid.ravel()])
    data += noise_strength * np.random.randn(*data.shape)
    return data



def create_figure1(fig, row):
    n_points = 20 
    lim = 4

    data = sample_perturbed_grid(range_x=(-lim, lim), range_y=(-lim, lim), num_x=n_points, num_y=n_points, noise_strength=0.1)
    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=400)

    X_pointwise = np.zeros((dg.n, 2))
    X_pointwise[:, 0] = 3
    X = VectorField.from_pointwise_basis(X_pointwise,dg) 

    Y_pointwise = np.zeros((dg.n, 2))
    Y_pointwise[:, 1] = data[:, 0] 
    Y = VectorField.from_pointwise_basis( Y_pointwise,dg)
    nabla_X_Y = cov_der(dg, X, Y)

    scale = 0.12
    line_width = 1
    arrow_scale = 0.3


    fig1 = plot_quiver_2d(data, X.to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)
    fig2 = plot_quiver_2d(data, Y.to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)
    fig3 = plot_quiver_2d(data, nabla_X_Y.to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)


    fig.add_traces(list(fig1.data), rows=row, cols=1)
    fig.add_traces(list(fig2.data), rows=row, cols=2)
    fig.add_traces(list(fig3.data), rows=row, cols=3)

   
def sample_annulus(center=(0.0, 0.0), grid_res=10, r=1, noise=0.0):
    cx, cy = center

    # Build grid spanning the bounding square of the disc
    xs = np.linspace(cx - r, cx + r, grid_res)
    ys = np.linspace(cy - r, cy + r, grid_res)
    X, Y = np.meshgrid(xs, ys)

    # Mask for points inside the disc
    mask = (X - cx)**2 + (Y - cy)**2 <= r**2

    # Return coordinates where mask=True
    points = np.column_stack((X[mask], Y[mask]))

    if noise > 0.0:
        points = points + np.random.normal(scale=noise, size=points.shape)


    return points


def create_figure2(fig, row):
    n_points = 1000
    data = sample_annulus(grid_res=30, r=1, noise=0.05)
    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=1000)

    rot_vf_pointwise = np.zeros((dg.n, 2))
    rot_vf_pointwise[:, 0] = -data[:, 1]
    rot_vf_pointwise[:, 1] = data[:, 0]
    rot_vf = VectorField.from_pointwise_basis(rot_vf_pointwise, dg)

    scale = 0.12
    line_width = 1
    arrow_scale = 0.3


    fig1 = plot_quiver_2d(data, rot_vf.to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)
    fig2 = plot_quiver_2d(data, rot_vf.to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)

    fig3 = plot_quiver_2d(data, (dg.levi_civita(rot_vf)(rot_vf)).to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)


    fig.add_traces(list(fig1.data), rows=row, cols=1)
    fig.add_traces(list(fig2.data), rows=row, cols=2)
    fig.add_traces(list(fig3.data), rows=row, cols=3)


def text_to_mask(text, grid_res=10, font_size=1.0, font_name="DejaVu Sans"):
    """Return (points, mask) arrays so we can progressively reveal sampled points."""
    fp = FontProperties(family=font_name)
    path = TextPath((0, 0), text, size=font_size, prop=fp)
    verts = path.vertices
    xmin, ymin = verts.min(axis=0)
    xmax, ymax = verts.max(axis=0)

    np.random.seed(0)



    xs = np.linspace(xmin, xmax, grid_res)
    ys = np.linspace(ymin, ymax, grid_res)
    X, Y = np.meshgrid(xs, ys)

    pts = np.column_stack((X.ravel(), Y.ravel()))

    #add some noise 
    # fix rng seed

    # np.random.seed(0)
    pts += 0.1 * np.random.randn(*pts.shape)

    # Handle holes properly (even-odd rule)
    codes = path.codes
    if codes is None:
        mask = path.contains_points(pts)
    else:
        polys = []
        current = []
        for v, c in zip(path.vertices, codes):
            if c == Path.MOVETO and current:
                polys.append(np.array(current))
                current = [v]
            else:
                current.append(v)
        polys.append(np.array(current))
        inside = np.zeros(len(pts), dtype=bool)
        for p in polys:
            poly = Path(p)
            inside ^= poly.contains_points(pts)
        mask = inside

    return pts, mask


def create_figure3(fig, row):
    n_points = 60
    text = r"$\nabla_X(Y)$"
    data, mask = text_to_mask(r"$\nabla_X(Y)$", grid_res=n_points, font_size=3.0, font_name="DejaVu Sans")
    data = data[mask]


    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=1000)

    scale = 0.16
    line_width = 0.5
    arrow_scale = 0.5

    X_pointwise = np.zeros((dg.n, 2))
    X_pointwise[:, 0] = 3
    X = VectorField.from_pointwise_basis(X_pointwise,dg) 

    Y_pointwise = np.zeros((dg.n, 2))
    Y_pointwise[:, 1] = 3
    Y = VectorField.from_pointwise_basis( Y_pointwise,dg)
    nabla_X_Y = cov_der(dg, X, Y)




    fig1 = plot_quiver_2d(data, X.to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)
    fig2 = plot_quiver_2d(data, Y.to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)
    fig3 = plot_quiver_2d(data, nabla_X_Y.to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)

    fig.add_traces(list(fig1.data), rows=row, cols=1)
    fig.add_traces(list(fig2.data), rows=row, cols=2)
    fig.add_traces(list(fig3.data), rows=row, cols=3)


import numpy as np

def sample_torus(n_theta: int, n_phi: int, R: float = 2.0, r: float = 0.7) -> np.ndarray:
    theta = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
    phi = np.linspace(0, 2*np.pi, n_phi, endpoint=False)
    theta, phi = np.meshgrid(theta, phi, indexing='ij')

    x = (R + r * np.cos(theta)) * np.cos(phi)
    y = (R + r * np.cos(theta)) * np.sin(phi)
    z = r * np.sin(theta)

    data = np.column_stack([x.ravel(), y.ravel(), z.ravel()])
    return data, theta.ravel(), phi.ravel()


def torus_frame(theta: np.ndarray, phi: np.ndarray, R: float, r: float) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Tangent directions
    e_theta = np.column_stack((
        -np.sin(theta) * np.cos(phi),
        -np.sin(theta) * np.sin(phi),
         np.cos(theta)
    ))

    e_phi = np.column_stack((
        -(R + r*np.cos(theta)) * np.sin(phi),
         (R + r*np.cos(theta)) * np.cos(phi),
         np.zeros_like(theta)
    ))

    # Normalize to get orthonormal frame
    e_theta /= np.linalg.norm(e_theta, axis=1, keepdims=True)
    e_phi /= np.linalg.norm(e_phi, axis=1, keepdims=True)

    # Normal vector (points outward from torus surface)
    n = np.column_stack((
        np.cos(theta) * np.cos(phi),
        np.cos(theta) * np.sin(phi),
        np.sin(theta)
    ))

    return e_theta, e_phi, n

def sec_curv(dg, X, Y): 
    term1 = dg.levi_civita(cov_der(dg, Y, Y))(X,X)
    term2 = dg.levi_civita(cov_der(dg, X, Y))(Y,X)
    term3 = dg.levi_civita(Y)(dg.lie_bracket(X, Y), X)

    res = term1 - term2 - term3
    res = res/ (dg.g(X, X) * dg.g(Y, Y) - dg.g(X, Y)**2)
    return res

def f(data):
    x, y, z = data[:, 0], data[:, 1], data[:, 2]
    return 1.0/(np.sqrt(x**2 + y**2))

def create_figure4(fig, row):

    r=0.5
    R=1.0
    
    data, theta, phi = sample_torus(n_theta=30, n_phi=30, R=R, r=r)

    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=200)

    e_theta, e_phi, n = torus_frame(theta, phi, R, r)

    e_theta = VectorField.from_pointwise_basis(e_theta, dg=dg)
    e_phi = VectorField.from_pointwise_basis(e_phi, dg=dg)

    scale = 0.05
    arrow_scale = 0.5
    line_width = 2

    fig1 = plot_quiver_3d(data, e_theta.to_ambient(), scale=0.6*scale, arrow_scale=3*arrow_scale, line_width=0.5*line_width)
    fig2 = plot_quiver_3d(data, e_phi.to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)
    fig3 = plot_quiver_3d(data, (cov_der(dg, e_phi, e_theta)).to_ambient(), scale=0.4*scale, arrow_scale=arrow_scale, line_width=line_width)
    # fig4 = plot_quiver_3d(data, (cov_der(dg, e_theta, e_phi)).to_ambient(), scale=0.4*scale, arrow_scale=arrow_scale)


    fig.add_traces(list(fig1.data), rows=row, cols=1)
    fig.add_traces(list(fig2.data), rows=row, cols=2)
    fig.add_traces(list(fig3.data), rows=row, cols=3)
    # fig.add_traces(list(fig4.data), rows=row, cols=4)

    # def func_vec_mult(dg, f, X):
    #     f_pointwise = f.to_pointwise_basis() 
    #     X_pointwise = X.to_pointwise_basis().reshape((-1, dg.dim))
    #     prod_pointwise = f_pointwise[..., None] * X_pointwise
    #     return VectorField.from_pointwise_basis(prod_pointwise, dg)

    # f_data = f(data)
    # f_func = Function.from_pointwise_basis(f_data, dg)

    # _ = func_vec_mult(dg, f_func, e_phi)

    # print(type(e_theta(f_func)))
    # theta_f_phi = func_vec_mult(dg, e_theta(f_func), e_phi)

    # fig5 = plot_scatter_3d(data, f_data)
    # fig6 = plot_quiver_3d(data, theta_f_phi.to_ambient(), scale=0.4*scale, arrow_scale=arrow_scale )
    
    # nabla_theta_phi = cov_der(dg, e_theta, e_phi)
    # f_times_nabla_theta_phi = func_vec_mult(dg, f_func, nabla_theta_phi)

    # fig7 = plot_quiver_3d(data, f_times_nabla_theta_phi.to_ambient(), scale=0.4*scale, arrow_scale=arrow_scale )

    # fig8 = plot_quiver_3d(data, (theta_f_phi+f_times_nabla_theta_phi).to_ambient(), scale=0.4*scale, arrow_scale=arrow_scale )

    # fig.add_traces(list(fig5.data), rows=2, cols=1)
    # fig.add_traces(list(fig6.data), rows=2, cols=2)
    # fig.add_traces(list(fig7.data), rows=2, cols=3)
    # fig.add_traces(list(fig8.data), rows=2, cols=4)



num_rows = 4
num_cols = 3

specs = [
    [{"type": "xy"}]*num_cols,
    [{"type": "xy"}]*num_cols,
    [{"type": "xy"}]*num_cols,
    [{"type": "scene"}]*num_cols,
    # [{"type": "scene"}]*num_cols,  
    # [{"type": "scene"}]*num_cols, 
    # [{"type": "scene"}]*num_cols,
    #  [{"type": "scene"}]*num_cols,
]

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.05,
    shared_xaxes=False,
    shared_yaxes=False
)

create_figure1(fig, row=1)
create_figure2(fig, row=2)
create_figure3(fig, row=3)
create_figure4(fig, row=4)


camera = dict(
    eye=dict(x=0, y=1.2, z=0.6),  
    up=dict(x=0, y=0, z=1),     
    center=dict(x=0, y=0, z=0)
)

# axis_range = dict(range=[-1.1, 1.1])
counter = 1
for r in range(num_rows-1):
    for c in range(num_cols):
        fig.update_xaxes(scaleanchor=f"y{counter}", scaleratio=1, row=r+1, col=c+1)
        fig.update_yaxes(scaleanchor=f"x{counter}", scaleratio=1, row=r+1, col=c+1)
        counter += 1


# for c in (1,2,3):
#     fig.update_xaxes(range=[-3.1, 3.1], row=1, col=c)

# for c in (1,2,3):
#     fig.update_xaxes(range=[-0.1, 1.1], row=2, col=c)


for c in (1,2,3):
    fig.update_xaxes(range=[-0.5, 8], row=3, col=c)

for i in range(1, num_rows * num_cols + 1):
    scene_key = f"scene{i}" if i > 1 else "scene"
    fig.update_layout({
        scene_key: dict(
            camera=camera,
            aspectmode="data",
            # xaxis=dict(**axis_range, title='x'),
            # yaxis=dict(**axis_range, title='y'),
            # zaxis=dict(range=[-2,2], title='z'),
        )
    })

fig.update_layout(width=900, height=1000) 


clean_fig(fig)
fig.update_layout(margin=dict(l=0, r=0, t=0, b=10))
fig.show()

fig.write_image("figs/levi_civita_connection.pdf", scale=1)



In [ ]:
label_list = [[r"X_1=\partial_x", r"Y_1=x\partial_y", r"\nabla_{X_1}(Y_1)=\partial_y"],
              [r"\mathrm{rot}= -y\partial_x + x \partial_y", r"\mathrm{rot}", r"\nabla_{\mathrm{rot}}(\mathrm{rot})=-(x\partial_x + y\partial_y)"],
              [r"\partial_x", r"\partial_y", r"\nabla_{\partial_x}(\partial_y)=0"],
              [r"e_{\theta}=\frac{1}{r}\partial_\theta", r"e_{\phi}=\frac{1}{R+r\cos(\theta)}\partial_\phi", r"\nabla_{e_\theta}(e_\phi)=\frac{-\sin(\theta)}{R+r\cos(\theta)}e_{\phi}"]
             ]

for r in range(len(label_list)):
    for c in range(len(label_list[r])):
        label_list[r][c] = "$" + label_list[r][c] + "$"

def labels(row, col):
    return label_list[row-1][col-1]

overpic_labels(fig, labels, 
                       stretch_x=0.97,
                       stretch_y=1.0,
                       offset_y=13)

# Difference Between Covariant and Lie Derivative

In [ ]:
def generate_figure_compare(fig):
    data = sample_perturbed_grid(range_x=(-4, 4), range_y=(-4, 4), num_x=21, num_y=21, noise_strength=0)
    dg = DiffusionGeometry.from_point_cloud(data, n_function_basis=400)

    

    X1_pointwise = np.zeros((data.shape[0], 2))
    X1_pointwise[:, 0] = 3
    X1 = VectorField.from_pointwise_basis(X1_pointwise, dg)

    Y1_pointwise = np.zeros((data.shape[0], 2))
    Y1_pointwise[:, 1] = data[:, 0]
    Y1 = VectorField.from_pointwise_basis(Y1_pointwise, dg)

    X2_pointwise = np.zeros((data.shape[0], 2))
    X2_pointwise[:, 0] = 3
    X2_pointwise[:, 1] = 0.8 * data[:, 1]
    X2 = VectorField.from_pointwise_basis(X2_pointwise, dg)

    Y2_pointwise = np.zeros((data.shape[0], 2))
    Y2_pointwise[:, 1] = data[:, 0]
    Y2 = VectorField.from_pointwise_basis(Y2_pointwise, dg)


    scale = 0.12
    line_width = 1
    arrow_scale = 0.3

    plot_vf = lambda vf: plot_quiver_2d(data, vf.to_ambient(), scale=scale, arrow_scale=arrow_scale, line_width=line_width)

    fig1 = plot_vf(X1)
    fig2 = plot_vf(Y1)
    fig3 = plot_vf(dg.levi_civita(Y1)(X1))
    fig4 = plot_vf(dg.lie_bracket(X1, Y1))

    fig5 = plot_vf(X2)
    fig6 = plot_vf(Y2)
    fig7 = plot_vf(dg.levi_civita(Y2)(X2))
    fig8 = plot_vf(dg.lie_bracket(X2, Y2))




    fig.add_traces(list(fig1.data), rows=1, cols=1)
    fig.add_traces(list(fig2.data), rows=1, cols=2)
    fig.add_traces(list(fig3.data), rows=1, cols=3)
    fig.add_traces(list(fig4.data), rows=1, cols=4)

    fig.add_traces(list(fig5.data), rows=2, cols=1)
    fig.add_traces(list(fig6.data), rows=2, cols=2)
    fig.add_traces(list(fig7.data), rows=2, cols=3)
    fig.add_traces(list(fig8.data), rows=2, cols=4)

    pass



num_rows = 2
num_cols = 4

specs = [
    [{"type": "xy"}]*num_cols,
    [{"type": "xy"}]*num_cols,
    # [{"type": "xy"}]*num_cols,
    # [{"type": "scene"}]*num_cols,
    # [{"type": "scene"}]*num_cols,  
    # [{"type": "scene"}]*num_cols, 
    # [{"type": "scene"}]*num_cols,
    #  [{"type": "scene"}]*num_cols,
]

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    specs=specs,
    horizontal_spacing=0.0,
    vertical_spacing=0.05,
    shared_xaxes=False,
    shared_yaxes=False
)

for r in range(1, num_rows + 1):
    for c in range(1, num_cols + 1):
        fig.add_shape(
            type="line",
            x0=-4, x1=4,
            y0=0, y1=0,
            line=dict(color="navy", width=2, dash="dot"),
            row=r, col=c,
            layer="below",
        )


generate_figure_compare(fig)



counter = 1
for r in range(num_rows):
    for c in range(num_cols):
        fig.update_xaxes(scaleanchor=f"y{counter}", scaleratio=1, row=r+1, col=c+1)
        fig.update_yaxes(scaleanchor=f"x{counter}", scaleratio=1, row=r+1, col=c+1)
        counter += 1

clean_fig(fig)

fig.update_layout(width=900, height=400) 

fig.update_layout(margin=dict(l=0, r=0, t=0, b=10))
fig.show()

fig.write_image("figs/levi_civita_connection_lie_bracket_compare.pdf", scale=1)



In [ ]:
label_list = [[r"X_1=\partial_x", r"Y_1=x\partial_y", r"\nabla_{X_1}(Y_1)", r"[X_1, Y_1]"],
              [r"X_2=\partial_x+0.5\partial_y", r"Y_2=x\partial_y", r"\nabla_{X_2}(Y_2)", r"[X_2, Y_2]"],
             ]

for r in range(len(label_list)):
    for c in range(len(label_list[r])):
        label_list[r][c] = "$" + label_list[r][c] + "$"

def labels(row, col):
    return label_list[row-1][col-1]

overpic_labels(fig, labels, 
                       stretch_x=0.97,
                       stretch_y=1.0,
                       offset_y=13)